# 08 — Robustez y evasión adversarial

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** verificar dos hipótesis críticas de seguridad:
1. El modelo de familias no depende de features manipulables del sistema operativo (`Init_Win_bytes_forward/backward`).
2. El modelo resiste (o no) ataques de evasión simples sobre el conjunto de test.

**Estructura:**
- Bloque 1: experimento sin `Init_Win_bytes`.
- Bloque 2: evasión adversarial controlada (3 variantes).
- Bloque 3: cierre del notebook y decisiones.

**Motivo de seguridad:** un modelo que depende de features trivialmente manipulables (ventana TCP del emisor) es evadible con un solo comando. Un modelo que no resiste perturbaciones pequeñas es inútil en producción.

In [ ]:
# ============================================
# Setup notebook 08
# ============================================
import os
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.types import NumericType
from functools import reduce
from pyspark.sql import DataFrame
import numpy as np
import gc

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}

try:
    drive.mount('/content/drive')
except:
    pass

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("VIGIA_Robustez") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()

print(f"✅ Spark {spark.version}")

## Bloque 1 — Modelo sin features de ventana TCP

En el notebook 07 detectamos que `Init_Win_bytes_forward` y `Init_Win_bytes_backward` dominan casi todas las clases en SHAP. Estas features representan el tamaño inicial de la ventana TCP y dependen fuertemente del sistema operativo del emisor.

**Hipótesis:** el modelo está aprendiendo en parte la "huella del SO" en lugar del comportamiento del ataque.

**Motivo de seguridad:** un atacante puede modificar el tamaño de ventana TCP con un solo comando. Si el modelo depende de esa feature, se vuelve trivialmente evadible.

In [ ]:
# ============================================
# Definir features excluyendo Init_Win_bytes
# ============================================
excluir = {"Label", "target_bin", "target_family", "Destination_Port"}
feature_cols = [
    c for c in train_df.columns
    if c not in excluir and isinstance(train_df.schema[c].dataType, NumericType)
]

# Nueva lista sin las features de ventana TCP
features_win = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]
feature_cols_no_win = [c for c in feature_cols if c not in features_win]

print(f"Features originales (sin puerto): {len(feature_cols)}")
print(f"Features tras quitar Init_Win_bytes: {len(feature_cols_no_win)}")
print(f"Eliminadas: {features_win}")

In [ ]:
# ============================================
# Mapeo familias + muestreo (idéntico al 06/07)
# ============================================
familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}

def aplicar_mapeo(df):
    expr = None
    for fam, idx in mapeo.items():
        cond = when(col("target_family") == fam, idx)
        expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
    return df.withColumn("target_family_idx", expr.otherwise(-1))

train_df = aplicar_mapeo(train_df)
val_df   = aplicar_mapeo(val_df)

print("Muestreando train al 50%...")
parts = [train_df.filter(col("target_family") == fam).sample(fraction=0.5, seed=42)
         for fam in familias]
train_sample = reduce(DataFrame.unionByName, parts)
train_sample.cache()

print(f"Train muestreado: {train_sample.count():,}")

In [ ]:
# ============================================
# Conversión a numpy
# ============================================
import time

print("Convirtiendo train...")
t0 = time.time()
train_pd = train_sample.select(feature_cols_no_win + ["target_family_idx"]).toPandas()
X_train = train_pd[feature_cols_no_win].values.astype(np.float32)
y_train = train_pd["target_family_idx"].values.astype(np.int8)
del train_pd; gc.collect()
print(f"✅ X_train: {X_train.shape} en {time.time()-t0:.1f}s")

print("\nConvirtiendo val...")
t0 = time.time()
val_pd = val_df.select(feature_cols_no_win + ["target_family_idx"]).toPandas()
X_val = val_pd[feature_cols_no_win].values.astype(np.float32)
y_val = val_pd["target_family_idx"].values.astype(np.int8)
del val_pd; gc.collect()
print(f"✅ X_val: {X_val.shape} en {time.time()-t0:.1f}s")

In [ ]:
# ============================================
# Entrenar modelo sin Init_Win_bytes
# ============================================
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

pesos = compute_sample_weight(class_weight="balanced", y=y_train)

modelo_no_win = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=7,
    eval_metric="mlogloss",
    max_depth=6,
    n_estimators=200,
    learning_rate=0.1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("Entrenando modelo sin Init_Win_bytes (5-8 min)...")
t0 = time.time()
modelo_no_win.fit(
    X_train, y_train,
    sample_weight=pesos,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"✅ Entrenado en {time.time()-t0:.1f}s")

# Evaluación rápida
y_pred_no_win = modelo_no_win.predict(X_val)
f1_macro_no_win = f1_score(y_val, y_pred_no_win, average="macro")
print(f"\nF1 macro sin Init_Win: {f1_macro_no_win:.4f}")

## Bloque 2 — Evasión adversarial controlada

**Objetivo:** simular 3 ataques de evasión simples que un atacante realista podría ejecutar sobre el modelo oficial.

**Variantes:**
1. **Cambio de ventana TCP:** factor aleatorio 0.5-2.0 sobre `Init_Win_bytes`.
2. **Suavizado temporal:** reducción del 30% en IAT Std.
3. **Combinada:** ambas perturbaciones.

**Método de evaluación:** recall por clase = proporción de muestras reales de clase i que se predicen como i. (No usar `average="macro"` sobre una máscara, da resultados erróneos.)

In [ ]:
# ============================================
# Preparar test y baseline
# ============================================
import xgboost as xgb
import numpy as np
import gc
from pyspark.sql.functions import col, when

# Cargar modelo oficial
modelo_oficial = xgb.XGBClassifier()
modelo_oficial.load_model(f"{REPO_LOCAL}/src/models/familias_sin_puerto.json")

# Cargar test y mapear
test_df_full = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")

expr = None
for fam, idx in mapeo.items():
    cond = when(col("target_family") == fam, idx)
    expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
test_df_full = test_df_full.withColumn("target_family_idx", expr.otherwise(-1))

# Feature list
excluir = {"Label", "target_bin", "target_family", "Destination_Port", "target_family_idx"}
feature_cols_ev = [
    c for c in test_df_full.columns
    if c not in excluir and isinstance(test_df_full.schema[c].dataType, NumericType)
]

# Filtrar solo las que el modelo espera
modelo_features = modelo_oficial.get_booster().feature_names
if modelo_features:
    feature_cols_ev = [c for c in feature_cols_ev if c in modelo_features]
print(f"Features usadas: {len(feature_cols_ev)}")

# Cargar a pandas
test_eval = test_df_full.filter("target_family_idx >= 0")
test_pd = test_eval.select(feature_cols_ev + ["target_family_idx"]).toPandas()
y_test = test_pd["target_family_idx"].values.astype(np.int8)
print(f"Test: {test_pd.shape}")

# Función auxiliar: recall por clase (CORREGIDA)
def recall_por_clase(y_true, y_pred):
    """Recall real por clase = proporción de muestras de clase i predichas como i."""
    resultado = {}
    for i, fam in enumerate(familias):
        mask = y_true == i
        if mask.sum() > 0:
            resultado[fam] = (y_pred[mask] == i).mean()
        else:
            resultado[fam] = None
    return resultado

# Baseline
X_base = test_pd[feature_cols_ev].values.astype(np.float32)
pred_base = modelo_oficial.predict(X_base)
recall_base = recall_por_clase(y_test, pred_base)

print("\n=== Recall baseline por clase ===")
for fam, r in recall_base.items():
    if r is not None:
        print(f"  {fam:15s} {r:.4f}")

## Variante 1: cambio de ventana TCP

Simula un atacante que modifica su ventana TCP inicial para parecer otro SO.

**Aplicación:** multiplicar `Init_Win_bytes_forward` y `Init_Win_bytes_backward` por un factor aleatorio entre 0.5 y 2.0.

In [ ]:
# ============================================
# Variante 1: cambio de ventana TCP
# ============================================
np.random.seed(42)

test_v1 = test_pd.copy()
factores = np.random.uniform(0.5, 2.0, size=len(test_v1))
for c in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
    if c in test_v1.columns:
        test_v1[c] = test_v1[c] * factores

pred_v1 = modelo_oficial.predict(test_v1[feature_cols_ev].values.astype(np.float32))
recall_v1 = recall_por_clase(y_test, pred_v1)

print("=== Variante 1: cambio de ventana TCP ===\n")
print(f"{'Clase':<15} {'Base':>10} {'V1':>10} {'Δ':>10}")
print("-" * 50)
for fam in familias:
    if recall_base[fam] is not None:
        d = recall_v1[fam] - recall_base[fam]
        flag = "⚠️" if d < -0.05 else "  "
        print(f"{flag} {fam:<13} {recall_base[fam]:>10.4f} {recall_v1[fam]:>10.4f} {d:>+10.4f}")

del test_v1; gc.collect()

## Variante 2: suavizado temporal

Simula un atacante que reduce la varianza de sus IATs para parecer tráfico más regular.

**Aplicación:** reducir `Flow_IAT_Std`, `Fwd_IAT_Std` y `Bwd_IAT_Std` un 30%.

In [ ]:
# ============================================
# Variante 2: suavizado temporal
# ============================================
test_v2 = test_pd.copy()
iat_cols = ["Flow_IAT_Std", "Fwd_IAT_Std", "Bwd_IAT_Std"]
iat_cols = [c for c in iat_cols if c in test_v2.columns]

for c in iat_cols:
    test_v2[c] = test_v2[c] * 0.7

pred_v2 = modelo_oficial.predict(test_v2[feature_cols_ev].values.astype(np.float32))
recall_v2 = recall_por_clase(y_test, pred_v2)

print("=== Variante 2: suavizado temporal ===\n")
print(f"{'Clase':<15} {'Base':>10} {'V2':>10} {'Δ':>10}")
print("-" * 50)
for fam in familias:
    if recall_base[fam] is not None:
        d = recall_v2[fam] - recall_base[fam]
        flag = "⚠️" if d < -0.05 else "  "
        print(f"{flag} {fam:<13} {recall_base[fam]:>10.4f} {recall_v2[fam]:>10.4f} {d:>+10.4f}")

del test_v2; gc.collect()

## Variante 3: ventana TCP + suavizado temporal

Escenario más realista: atacante que combina ambas perturbaciones.

In [ ]:
# ============================================
# Variante 3: ventana + suavizado temporal
# ============================================
np.random.seed(42)

test_v3 = test_pd.copy()
factores = np.random.uniform(0.5, 2.0, size=len(test_v3))
for c in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
    if c in test_v3.columns:
        test_v3[c] = test_v3[c] * factores
for c in iat_cols:
    test_v3[c] = test_v3[c] * 0.7

pred_v3 = modelo_oficial.predict(test_v3[feature_cols_ev].values.astype(np.float32))
recall_v3 = recall_por_clase(y_test, pred_v3)

print("=== Variante 3: ventana + temporal ===\n")
print(f"{'Clase':<15} {'Base':>10} {'V3':>10} {'Δ':>10}")
print("-" * 50)
for fam in familias:
    if recall_base[fam] is not None:
        d = recall_v3[fam] - recall_base[fam]
        flag = "⚠️" if d < -0.05 else "  "
        print(f"{flag} {fam:<13} {recall_base[fam]:>10.4f} {recall_v3[fam]:>10.4f} {d:>+10.4f}")

del test_v3; gc.collect()

In [ ]:
# ============================================
# Bloque 2 completo: baseline + 3 variantes + guardar
# ============================================
import os
import numpy as np
import gc
import json
import xgboost as xgb
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.types import NumericType
from google.colab import drive

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
else:
    %cd {REPO_LOCAL}

try:
    drive.mount('/content/drive')
except:
    pass

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("VIGIA_evasion") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()

familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}

# --- Cargar modelo oficial ---
modelo_oficial = xgb.XGBClassifier()
modelo_oficial.load_model(f"{REPO_LOCAL}/src/models/familias_sin_puerto.json")
print("✅ Modelo cargado")

# --- Cargar test ---
test_df_full = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")

expr = None
for fam, idx in mapeo.items():
    cond = when(col("target_family") == fam, idx)
    expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
test_df_full = test_df_full.withColumn("target_family_idx", expr.otherwise(-1))

# --- Features: reconstruir manualmente (mismo criterio que 06/07) ---
excluir = {"Label", "target_bin", "target_family", "Destination_Port", "target_family_idx"}
feature_cols_ev = [
    c for c in test_df_full.columns
    if c not in excluir and isinstance(test_df_full.schema[c].dataType, NumericType)
]
print(f"Features usadas: {len(feature_cols_ev)}")

# Verificar que coincide con las 77 esperadas
assert len(feature_cols_ev) == 77, f"Se esperaban 77 features, hay {len(feature_cols_ev)}"

# --- Convertir a pandas ---
test_eval = test_df_full.filter("target_family_idx >= 0")
test_pd = test_eval.select(feature_cols_ev + ["target_family_idx"]).toPandas()
y_test = test_pd["target_family_idx"].values.astype(np.int8)
print(f"Test: {test_pd.shape}")

# --- Función recall por clase ---
def recall_por_clase(y_true, y_pred):
    resultado = {}
    for i, fam in enumerate(familias):
        mask = y_true == i
        if mask.sum() > 0:
            resultado[fam] = float((y_pred[mask] == i).mean())
        else:
            resultado[fam] = None
    return resultado

# --- Baseline ---
X_base = test_pd[feature_cols_ev].values.astype(np.float32)
pred_base = modelo_oficial.predict(X_base)
recall_base = recall_por_clase(y_test, pred_base)
print("\n=== Baseline ===")
for fam, r in recall_base.items():
    if r is not None:
        print(f"  {fam:15s} {r:.4f}")

# --- Variante 1: ventana TCP ---
np.random.seed(42)
test_v1 = test_pd.copy()
factores = np.random.uniform(0.5, 2.0, size=len(test_v1))
for c in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
    if c in test_v1.columns:
        test_v1[c] = test_v1[c] * factores
pred_v1 = modelo_oficial.predict(test_v1[feature_cols_ev].values.astype(np.float32))
recall_v1 = recall_por_clase(y_test, pred_v1)
del test_v1; gc.collect()

# --- Variante 2: suavizado temporal ---
test_v2 = test_pd.copy()
iat_cols = ["Flow_IAT_Std", "Fwd_IAT_Std", "Bwd_IAT_Std"]
iat_cols = [c for c in iat_cols if c in test_v2.columns]
for c in iat_cols:
    test_v2[c] = test_v2[c] * 0.7
pred_v2 = modelo_oficial.predict(test_v2[feature_cols_ev].values.astype(np.float32))
recall_v2 = recall_por_clase(y_test, pred_v2)
del test_v2; gc.collect()

# --- Variante 3: combinada ---
np.random.seed(42)
test_v3 = test_pd.copy()
factores = np.random.uniform(0.5, 2.0, size=len(test_v3))
for c in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
    if c in test_v3.columns:
        test_v3[c] = test_v3[c] * factores
for c in iat_cols:
    test_v3[c] = test_v3[c] * 0.7
pred_v3 = modelo_oficial.predict(test_v3[feature_cols_ev].values.astype(np.float32))
recall_v3 = recall_por_clase(y_test, pred_v3)
del test_v3; gc.collect()

# --- Tabla resumen ---
print("\n=== Comparativa de evasión ===\n")
print(f"{'Clase':<15} {'Base':>8} {'V1 win':>9} {'V2 iat':>9} {'V3 comb':>9}")
print("-" * 55)
for fam in familias:
    if recall_base[fam] is not None:
        print(f"{fam:<15} {recall_base[fam]:>8.4f} {recall_v1[fam]:>9.4f} "
              f"{recall_v2[fam]:>9.4f} {recall_v3[fam]:>9.4f}")

# --- Guardar JSON ---
resultados_evasion = {
    fam: {
        "base": recall_base[fam],
        "v1_ventana": recall_v1[fam],
        "v2_iat": recall_v2[fam],
        "v3_combinada": recall_v3[fam],
    }
    for fam in familias if recall_base[fam] is not None
}

MODELS_DIR = f"{REPO_LOCAL}/src/models"
with open(f"{MODELS_DIR}/evasion_resultados.json", "w") as f:
    json.dump(resultados_evasion, f, indent=2)

print(f"\n✅ Resultados guardados en {MODELS_DIR}/evasion_resultados.json")

In [ ]:
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Evasión adversarial: resultado crítico

**3 variantes probadas sobre test (424.532 muestras):**
- V1 — Cambio de ventana TCP: factor aleatorio 0.5-2.0 sobre `Init_Win_bytes_forward/backward`.
- V2 — Suavizado temporal: reducción del 30% en `Flow_IAT_Std`, `Fwd_IAT_Std`, `Bwd_IAT_Std`.
- V3 — Combinada.

**Resultados (recall por clase):**

| Clase | Base | V1 ventana | V2 IAT | V3 combinada |
|---|---|---|---|---|
| BENIGN | 0.9984 | 0.9989 | 0.9985 | 0.9990 |
| DoS | 0.9997 | 0.3742 | 0.9960 | 0.3690 |
| Probe | 0.9998 | 0.9933 | 0.9997 | 0.9931 |
| DDoS | 0.9999 | 0.5204 | 0.9999 | 0.4911 |
| Brute-force | 0.9990 | 0.4071 | 0.9990 | 0.4066 |
| Web | 0.9851 | 0.3164 | 0.9851 | 0.3045 |
| Bot | 0.9839 | 0.6367 | 0.9839 | 0.6367 |

**Análisis:**
1. **V2 (suavizado temporal) no tiene impacto significativo.** El modelo no depende de la varianza de IAT para clasificar.
2. **V1 (cambio de ventana TCP) es devastador.** 5 de 7 clases caen por debajo del 0.65 de recall.
3. **V3 (combinada) ≈ V1.** La caída se debe exclusivamente a la ventana TCP.
4. **BENIGN y Probe resisten.** Coherente con el experimento sin Init_Win_bytes del notebook 07: eran las dos únicas clases que no dependían de esa feature.

**Implicación de seguridad (crítica):**
Un atacante puede modificar su ventana TCP inicial con un comando trivial en cualquier SO (Linux: `ip route change`, Windows: modificación del stack TCP). Con un solo cambio puede pasar de recall 0.9997 a 0.3742 en DoS, de 0.9999 a 0.5204 en DDoS, de 0.9851 a 0.3164 en Web. El detector es evadible sin conocer los pesos del modelo, solo conociendo qué feature usa.

**Comparación con la literatura:**
Los papers que reportan F1 > 0.99 sobre CIC-IDS2017 con `Init_Win_bytes` incluida no están midiendo detección de ataques. Están midiendo la huella del sistema operativo del generador del tráfico. Este proyecto documenta esa distinción como aporte principal.

**Decisiones:**
- El resultado se convierte en el titular de la sección de conclusiones.
- Se documenta como limitación crítica del modelo oficial.
- Justifica el uso de detección no supervisada complementaria como línea de trabajo futuro.
- El modelo sin Init_Win_bytes (F1 0.66) se presenta como alternativa honesta, aunque con menor rendimiento.

**Próximos pasos:**
- Bloque 3: dashboard con SHAP integrado.
- Memoria: este resultado va al capítulo de discusión y conclusiones.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

!git add .
!git commit -m "Evasión adversarial: hallazgo crítico (recall cae al 37% en DoS con cambio de ventana TCP)"

# Push
from getpass import getpass
TOKEN = getpass("Pega tu token: ")
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push